# ChemCoTBench-V2：20题原始化学审计

AI审读，不是人类专家金标准。仅审原始数据；无Poe调用。

在本文件目录用已安装RDKit的Python运行。保存输出由普通Python顺序执行器生成，未通过Jupyter内核。

In [1]:
from pathlib import Path
import json
from collections import Counter
from audit import run, HERE
from verify_audit import verify
evidence = run()


{
  "sample_size": 20,
  "failed_numeric_or_answer_checks": 0,
  "reconstructed_answers": 20,
  "ids": [
    "mol_edit.add_v2.0022",
    "mol_edit.add_v2.0079",
    "mol_edit.add_v2.0101",
    "mol_edit.add_v2.0150",
    "mol_edit.add_v2.0185",
    "mol_edit.add_v2.0274",
    "mol_edit.add_v2.0295",
    "mol_edit.delete_v2.0028",
    "mol_edit.delete_v2.0056",
    "mol_edit.delete_v2.0107",
    "mol_edit.delete_v2.0126",
    "mol_edit.delete_v2.0186",
    "mol_edit.delete_v2.0255",
    "mol_edit.delete_v2.0287",
    "mol_edit.substitute_v2.0005",
    "mol_edit.substitute_v2.0009",
    "mol_edit.substitute_v2.0043",
    "mol_edit.substitute_v2.0065",
    "mol_edit.substitute_v2.0078",
    "mol_edit.substitute_v2.0239"
  ]
}


In [2]:
verification = verify()


{
  "sample_records": 20,
  "original_steps": 106,
  "numeric_structure_checks_passed": 246,
  "exact_graph_reconstructions": 20,
  "verbatim_finding_quotes_verified": 4,
  "source_files_hash_unchanged": 6,
  "deliberate_error_detection_tests_passed": 3,
  "scaffold_counterexample_verified": true,
  "live_poe_requests": 0
}


## 全部20题数值证据

图重建读取原FORMAL指定的编辑，不能单独证明instruction语义；后者对应独立审读记录。

In [3]:
for r in evidence['records']:
    print(r['origin_id'], 'source=', r['source_computed'], 'answer=', r['answer_computed'], 'reconstruction=', r['graph_reconstruction'])


mol_edit.add_v2.0022 source= {'heavy_atoms': 36, 'rings': 4, 'formula': 'C25H31N5O5S', 'elements': {'C': 25, 'O': 5, 'N': 5, 'S': 1}, 'formal_charge': 0} answer= {'heavy_atoms': 42, 'rings': 4, 'formula': 'C28H37N5O7S2', 'elements': {'C': 28, 'S': 2, 'O': 7, 'N': 5}, 'formal_charge': 0} reconstruction= {'matching_removal_sets': 1, 'valid_candidate_count': 4, 'exact_isomeric_answer_recovered': True, 'answer_matches': [{'removed_maps': [], 'fragment_attachment_index_0_based': 3, 'fragment_attachment_element': 'S'}], 'candidates': ['CCC(NCCCOc1nc(C2CC2)nc(NS(=O)(=O)c2ccc(C(C)C)cn2)c1Oc1ccccc1OC)[SH](=O)=O', 'CCCS(=O)(=O)NCCCOc1nc(C2CC2)nc(NS(=O)(=O)c2ccc(C(C)C)cn2)c1Oc1ccccc1OC', 'COc1ccccc1Oc1c(NS(=O)(=O)c2ccc(C(C)C)cn2)nc(C2CC2)nc1OCCCNC(C)C[SH](=O)=O', 'COc1ccccc1Oc1c(NS(=O)(=O)c2ccc(C(C)C)cn2)nc(C2CC2)nc1OCCCNCCC[SH](=O)=O'], 'scope': 'single boundary-bond removal (order 1/2/3) / single-bond addition; valence-aware H capping, atom identity and stereochemical graph retained; not a reac

## 逐题AI语义审读

这些判断来自原题、结构和完整正文的审读，不是由PASS标记自动推断。剩余16题未发现明确问题，不是逐token真值认证。

In [4]:
review = json.loads((HERE / 'review.json').read_text())
print(json.dumps(review, ensure_ascii=False, indent=2))
print('Observed categories, not population error rates:', dict(Counter(r['category'] for r in review['records'])))


{
  "reviewer": "AI-assisted chemistry review; not independent human expert annotation",
  "scope": "20 original origins, 106 steps; no generated H/N used; no Poe calls",
  "categories": {
    "factual_error": "明确的正文化学事实错误，不代表产物错误",
    "incomplete_reasoning": "正文未实例化，不属于已断言的错误事实",
    "mechanism_caveat": "净图编辑可成立，但不能据此确定实验机制或原子来源",
    "no_material_issue_found": "限定检查中未发现明确问题，不等于逐 token 真值认证"
  },
  "findings": [
    {
      "id": "F1",
      "origin_id": "mol_edit.add_v2.0150",
      "steps": [
        5
      ],
      "category": "factual_error",
      "confidence": "high",
      "severity": "medium; important for negative-label contamination",
      "quote": "Source has 6 rings (pyrazolo[1,5-a]pyrimidine core, pyrrolidine, indazole core, and phenyl). Product has 6 rings. RING_DELTA = 0.",
      "explanation": "总环数6与产物均正确，但第一处稠合杂环名称不符。原结构六元环 maps=[2,31,6,5,4,3] 含3个N；与之稠合的五元环 maps=[13,7,6,31,14] 含1个N。pyrazolo[1,5-a]pyrimidine 则是六元环2N、五元环2N；两者总N都可为3，所以只检查总元素数或总环数会漏掉这个错误。",
      "reco

## 限制与来源

仅20/150本地已筛选题；没有上游错误率估计或固定format性能实验。

[PubChem骨架参照](https://pubchem.ncbi.nlm.nih.gov/compound/Pyrazolo_1_5-a_pyrimidine)；[OpenStax酯化与水解](https://openstax.org/books/organic-chemistry/pages/21-6-chemistry-of-esters)；[RDKit环计数](https://www.rdkit.org/docs/RDKit_Book.html#ring-finding-and-sssr)。完整边界与处置建议见REPORT.md。